In [1]:
import numpy as np
import os
import pandas as pd


/its/home/nn268/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


### read in csv

In [2]:
# data files

# trimed paths
trimPath = "RealAnts_ALL_TRIM.csv"
# trimed paths no gap
ngPath = "RealAnts_ALL_TRIMNG.csv"
# chunked path


In [23]:
all_paths_trim = pd.read_csv(trimPath)

all_paths_ng = pd.read_csv(ngPath)

/tmp/ipykernel_45234/179210603.py:1: DtypeWarning: Columns (4,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  all_paths_trim = pd.read_csv(trimPath)
/tmp/ipykernel_45234/179210603.py:3: DtypeWarning: Columns (4,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  all_paths_ng = pd.read_csv(ngPath)


In [62]:
list(all_paths_trim)

['Date',
 'ID',
 'frame',
 'x',
 'y',
 'Nest',
 'Condition',
 'Sector',
 'Direction',
 'DI',
 'rad',
 'theta',
 'dTheta',
 'thetaR2N',
 'deltaDist',
 'accumDist',
 'speed',
 'time',
 'meanSpeed',
 'dx',
 'dy',
 'trim_StraightnessIDX',
 'trim_beeline',
 'trim_circR']

### group by nest

In [24]:
all_paths_trim = all_paths_trim.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'index'])
all_paths_ng = all_paths_ng.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'index'])

all_paths_trim_Wigwam = all_paths_trim[all_paths_trim['Nest']=='Wigwam']
all_paths_trim_NewNest = all_paths_trim[all_paths_trim['Nest']=='NewNest']

all_paths_ng_Wigwam = all_paths_ng[all_paths_ng['Nest']=='Wigwam']
all_paths_ng_newNest= all_paths_ng[all_paths_ng['Nest']=='NewNest']

In [28]:
all_paths_trim_NewNest[:5]

,Date,ID,frame,x,y,Nest,Condition,Sector,Direction,DI,...,deltaDist,accumDist,speed,time,meanSpeed,dx,dy,trim_StraightnessIDX,trim_beeline,trim_circR
407115,Sep-20,108.0,137.0,25.870527,-16.787486,NewNest,Minus 3m ZV,9.0,180.0,1.0,...,1.673954,NaN,NaN,1.37,6.77057,NaN,NaN,0.807712,265.698065,0.492916
407116,Sep-20,108.0,138.0,27.006229,-16.722622,NewNest,Minus 3m ZV,9.0,180.0,1.0,...,1.137553,1.137553,11.375531,1.38,6.77057,1.135702,0.064863,0.807712,265.698065,0.492916
407117,Sep-20,108.0,139.0,28.440841,-17.022042,NewNest,Minus 3m ZV,9.0,180.0,1.0,...,1.465525,2.603078,14.655246,1.39,6.77057,1.434611,-0.299420,0.807712,265.698065,0.492916
407118,Sep-20,108.0,140.0,29.457926,-17.792296,NewNest,Minus 3m ZV,9.0,180.0,1.0,...,1.275834,3.878912,12.758344,1.40,6.77057,1.017086,-0.770253,0.807712,265.698065,0.492916
407119,Sep-20,108.0,141.0,30.542514,-18.484708,NewNest,Minus 3m ZV,9.0,180.0,1.0,...,1.286766,5.165678,12.867660,1.41,6.77057,1.084588,-0.692413,0.807712,265.698065,0.492916


### group by condition

In [152]:
def get_meaan_var_std(df, target:str, unit:str):
    import csv
    import os
    
    conditions = np.unique(df['Condition'])
    statsDicts = []
    if not os.path.exists(os.path.join('AntStats_150126.csv')):
        print('No File Found!')
        
        with open('AntStats_150126.csv', 'w') as file:
            fieldnames = ['Nest', 'Condition', 'Metric', 'Mean', 'StdDev', 'N']
            writer = csv.DictWriter(file,  fieldnames=fieldnames)
            writer.writeheader()
        print('headers Added')
        
    for cond in conditions:
        TheMissing = 0
        usedf = df[df['Condition'] == cond]
        N = len(np.unique(usedf['ID'].astype(str)))
        
        #strings = []
        #for s in all_paths_ng_newNest[target]:
        #    try:
        #        s = float(s)
        #        #ints.append(s)
        #    except:
        #        strings.append(s)
        #usedf[target] = 
        usedf.loc[:,target] = pd.to_numeric(usedf[target], errors='coerce')
        TheMissing = usedf[target].isna().sum()
        #print(f"The Missing {TheMissing}")
        usedf.dropna(subset = [target])

        
        #usedf[target] = usedf[target].astype(float)
        mean_target = pd.to_numeric(usedf[target]).mean()
        #print(f"{np.unique(df['Nest'])[0]}  {cond}     mean speed {np.mean(usedf['speed'])}     varience {np.var(usedf['speed'])}      stdev {np.std(usedf['speed'])}")
        #print(f"                          mean speed {mean_speed}")
        print(f"{np.unique(df['Nest'])[0]}  {cond}   MEAN {target.upper()} : {np.mean(usedf[target])} \u00B1 {np.std(usedf[target])} {unit} (Mean \u00B1 SD, n = {N})")
        #print(u"\u00B1")
        print("")
        stats_dict = {'Nest': np.unique(df['Nest'])[0], 'Condition' : cond, 'Metric' : target, 'Mean': np.mean(usedf[target]), 'StdDev': np.std(usedf[target]), 'N': N}
        statsDicts.append(stats_dict)
        # WRITE TO CSV
        # Turn data into a dictionary
    #print(statsDicts)
    with open('AntStats_150126.csv', 'a') as file:
        fieldnames = ['Nest', 'Condition', 'Metric', 'Mean', 'StdDev', 'N']
        writer = csv.DictWriter(file,  fieldnames=fieldnames)
        #writer.writeheader()
        writer.writerows(statsDicts)


#get_meaan_var_std(all_paths_ng_Wigwam, target = "speed", unit = 'cm/s')

In [117]:
list(all_paths_ng_newNest)

['Date',
 'ID',
 'frame',
 'x',
 'y',
 'Nest',
 'Condition',
 'Sector',
 'Direction',
 'DI',
 'rad',
 'theta',
 'dTheta',
 'thetaR2N',
 'deltaDist',
 'accumDist',
 'speed',
 'time',
 'meanSpeed',
 'dx',
 'dy',
 'trimNG_StraightnessIDX',
 'trimNG_beeline',
 'trimNG_circR']

In [113]:
strings = []
ints = []

for s in all_paths_ng_newNest['Sector']:
    try:
        s = int(s)
        ints.append(s)
    except:
        strings.append(s)
    

print(np.unique(strings))
print(np.unique(ints))

TypeError: Cannot index by location index with a non-integer key

In [153]:
get_meaan_var_std(all_paths_ng_newNest, target = "speed", unit = 'cm/s')
print("")
get_meaan_var_std(all_paths_ng_newNest, target = "trimNG_StraightnessIDX", unit = "")
print("")
get_meaan_var_std(all_paths_ng_newNest, target = "trimNG_circR", unit = "")
print("")
get_meaan_var_std(all_paths_ng_newNest, target = "Sector", unit = "")
print("")
get_meaan_var_std(all_paths_ng_newNest, target = "Direction", unit = "")



NewNest  Minus 3m ZV   MEAN SPEED : 2.9747541389975067 ± 3.6247387683040855 cm/s (Mean ± SD, n = 36)

NewNest  Off-route 1 ZV   MEAN SPEED : 4.13259120003699 ± 4.755639948428915 cm/s (Mean ± SD, n = 65)

NewNest  On-route 1 ZV   MEAN SPEED : 4.257099658688812 ± 4.655327178392806 cm/s (Mean ± SD, n = 69)

NewNest  On-route 1B ZV   MEAN SPEED : 1.9529590982703258 ± 3.1047658542799694 cm/s (Mean ± SD, n = 64)

NewNest  On-route 2 ZV   MEAN SPEED : 2.959814815370135 ± 3.971120712728742 cm/s (Mean ± SD, n = 66)

NewNest  Plus 3m ZV   MEAN SPEED : 2.2298863346583984 ± 2.943209616839984 cm/s (Mean ± SD, n = 34)

NewNest  Route 0m ZV   MEAN SPEED : 2.201031322670906 ± 3.0509140764003644 cm/s (Mean ± SD, n = 43)

NewNest  Unfamiliar FV   MEAN SPEED : 2.033787251533258 ± 3.0486081245773415 cm/s (Mean ± SD, n = 61)

NewNest  Unfamiliar ZV   MEAN SPEED : 1.8916024852563953 ± 3.1077151740645643 cm/s (Mean ± SD, n = 69)

NewNest  Unfamiliar ZV 2   MEAN SPEED : 1.6069140370619917 ± 2.4270414099411006

In [154]:
get_meaan_var_std(all_paths_ng_Wigwam, target = "speed", unit = 'cm/s')
print("")
get_meaan_var_std(all_paths_ng_Wigwam, target = "trimNG_StraightnessIDX", unit = "")
print("")
get_meaan_var_std(all_paths_ng_Wigwam, target = "trimNG_circR", unit = "")
print("")
get_meaan_var_std(all_paths_ng_Wigwam, target = "Sector", unit = "")
print("")
get_meaan_var_std(all_paths_ng_Wigwam, target = "Direction", unit = "")

Wigwam  Off-route 1 ZV   MEAN SPEED : 3.8050328475318853 ± 5.5125620966572955 cm/s (Mean ± SD, n = 49)

Wigwam  On-route 1 ZV   MEAN SPEED : 4.4834813387105354 ± 5.794381426951653 cm/s (Mean ± SD, n = 48)

Wigwam  On-route 2 ZV   MEAN SPEED : 4.1690427411517685 ± 5.0239944479776035 cm/s (Mean ± SD, n = 52)

Wigwam  Unfamiliar FV   MEAN SPEED : 3.062830561148166 ± 4.573198336078641 cm/s (Mean ± SD, n = 53)

Wigwam  Unfamiliar ZV   MEAN SPEED : 3.554691194128304 ± 4.650532017285899 cm/s (Mean ± SD, n = 53)


Wigwam  Off-route 1 ZV   MEAN TRIMNG_STRAIGHTNESSIDX : 0.33464484897588165 ± 0.19343012939964438  (Mean ± SD, n = 49)

Wigwam  On-route 1 ZV   MEAN TRIMNG_STRAIGHTNESSIDX : 0.40462596359527503 ± 0.2706788278529638  (Mean ± SD, n = 48)

Wigwam  On-route 2 ZV   MEAN TRIMNG_STRAIGHTNESSIDX : 0.33441762309450396 ± 0.20947045963732305  (Mean ± SD, n = 52)

Wigwam  Unfamiliar FV   MEAN TRIMNG_STRAIGHTNESSIDX : 0.3523729254359445 ± 0.19255181349362277  (Mean ± SD, n = 53)

Wigwam  Unfamilia

In [ ]:
all_paths_trim_Wigwam
all_paths_ng_Wigwam